# neuropt — LLM-guided ML optimization

<a target="_blank" href="https://colab.research.google.com/github/loevlie/neuropt/blob/main/examples/neuropt_demo.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

**An LLM reads your training curves and designs your next experiment.**

neuropt uses an LLM to optimize hyperparameters and architectures. Unlike Optuna or random search, it sees *per-epoch training curves* — so it can spot overfitting, underfitting, and learning rate issues, then reason about what to try next.

This notebook shows two ways to use it:
1. **Define a search space** — you control what gets tuned
2. **Give it a model** — neuropt figures out what to tune

[![GitHub](https://img.shields.io/github/stars/loevlie/neuropt?style=social)](https://github.com/loevlie/neuropt)
[![PyPI](https://img.shields.io/pypi/v/neuropt)](https://pypi.org/project/neuropt/)

In [ ]:
!pip install -q "neuropt[llm,torch]"

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
    print("✓ API key loaded from Colab Secrets")
except Exception:
    if not os.environ.get("ANTHROPIC_API_KEY"):
        import getpass
        os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("ANTHROPIC_API_KEY: ")
        print("✓ API key set")

## Option 1: Define a search space

You write a training function + tell neuropt what to search. It handles the rest.

In [ ]:
import math
import numpy as np
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader, Subset

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

# Load FashionMNIST (small subset for fast iteration)
transform = T.Compose([T.Grayscale(3), T.Resize(32), T.ToTensor(), T.Normalize([0.5]*3, [0.5]*3)])
train_ds = torchvision.datasets.FashionMNIST("./data", train=True, download=True, transform=transform)
val_ds = torchvision.datasets.FashionMNIST("./data", train=False, download=True, transform=transform)
rng = np.random.default_rng(0)
train_loader = DataLoader(Subset(train_ds, rng.choice(len(train_ds), 5000, replace=False)),
                          batch_size=128, shuffle=True, num_workers=2)
val_loader = DataLoader(Subset(val_ds, rng.choice(len(val_ds), 1250, replace=False)),
                        batch_size=128, shuffle=False, num_workers=2)

In [ ]:
ACTIVATIONS = {"relu": nn.ReLU, "gelu": nn.GELU, "leaky_relu": nn.LeakyReLU, "silu": nn.SiLU}
EPOCHS = 10

class ConfigurableCNN(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        act = ACTIVATIONS[cfg["activation"]]
        self.blocks, self.projs, self.pools = nn.ModuleList(), nn.ModuleList(), nn.ModuleList()
        self.use_res = cfg["use_residual"]
        prev, spatial = 3, 32

        for i in range(cfg["n_blocks"]):
            out = min(512, max(8, int(cfg["base_channels"] * cfg["channel_growth"] ** i)))
            layers = [nn.Conv2d(prev, out, cfg["kernel_size"], padding=cfg["kernel_size"]//2, bias=not cfg["use_batchnorm"])]
            if cfg["use_batchnorm"]: layers.append(nn.BatchNorm2d(out))
            layers.append(act())
            if cfg["dropout"] > 0: layers.append(nn.Dropout2d(cfg["dropout"]))
            self.blocks.append(nn.Sequential(*layers))
            self.projs.append(nn.Conv2d(prev, out, 1, bias=False) if self.use_res and prev != out
                              else (nn.Identity() if self.use_res else None))
            if (i+1) % cfg["pool_every"] == 0 and spatial > 2:
                self.pools.append(nn.MaxPool2d(2) if cfg["pool_type"] == "max" else nn.AvgPool2d(2))
                spatial //= 2
            else:
                self.pools.append(None)
            prev = out

        self.gap = nn.AdaptiveAvgPool2d(1)
        self.head = (nn.Sequential(nn.Linear(prev, cfg["fc_hidden"]), act(), nn.Dropout(cfg["dropout"]),
                                   nn.Linear(cfg["fc_hidden"], 10))
                     if cfg["fc_hidden"] > 0 else nn.Linear(prev, 10))

    def forward(self, x):
        for blk, proj, pool in zip(self.blocks, self.projs, self.pools):
            h = blk(x)
            if self.use_res and proj is not None: h = h + proj(x)
            x = pool(h) if pool else h
        return self.head(self.gap(x).flatten(1))


def train_fn(config):
    try:
        model = ConfigurableCNN(config).to(DEVICE)
    except Exception as e:
        return {"score": float("inf"), "status": "build_error", "error": str(e)}

    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    opt_name = config.get("optimizer", "adamw")
    if opt_name == "sgd":
        opt = torch.optim.SGD(model.parameters(), lr=config["lr"], momentum=0.9, weight_decay=config["wd"])
    elif opt_name == "adam":
        opt = torch.optim.Adam(model.parameters(), lr=config["lr"], weight_decay=config["wd"])
    else:
        opt = torch.optim.AdamW(model.parameters(), lr=config["lr"], weight_decay=config["wd"])

    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
    crit = nn.CrossEntropyLoss()
    train_losses, val_losses, val_accuracies = [], [], []

    for _ in range(EPOCHS):
        model.train()
        tl, tn = 0.0, 0
        for imgs, tgts in train_loader:
            imgs, tgts = imgs.to(DEVICE), tgts.to(DEVICE)
            opt.zero_grad()
            loss = crit(model(imgs), tgts)
            if math.isnan(loss.item()):
                return {"score": float("inf"), "status": "nan", "n_params": n_params,
                        "train_losses": train_losses, "val_losses": val_losses, "val_accuracies": val_accuracies}
            loss.backward(); opt.step()
            tl += loss.item() * imgs.size(0); tn += imgs.size(0)
        sched.step(); train_losses.append(tl / tn)

        model.eval()
        vl, vn, vc = 0.0, 0, 0
        with torch.no_grad():
            for imgs, tgts in val_loader:
                imgs, tgts = imgs.to(DEVICE), tgts.to(DEVICE)
                out = model(imgs)
                vl += crit(out, tgts).item() * imgs.size(0)
                vc += (out.argmax(1) == tgts).sum().item(); vn += imgs.size(0)
        val_losses.append(vl / vn); val_accuracies.append(vc / vn)

    return {"score": val_losses[-1], "accuracy": val_accuracies[-1], "n_params": n_params,
            "train_losses": train_losses, "val_losses": val_losses, "val_accuracies": val_accuracies}

print(f"✓ Model + train_fn defined ({EPOCHS} epochs per eval)")

In [ ]:
from neuropt import ArchSearch

search_space = {
    "n_blocks":       (2, 8),
    "base_channels":  (16, 128),
    "channel_growth": (1.0, 2.5),
    "kernel_size":    [3, 5],
    "activation":     ["relu", "gelu", "leaky_relu", "silu"],
    "use_residual":   [True, False],
    "use_batchnorm":  [True, False],
    "dropout":        (0.0, 0.5),
    "pool_every":     (1, 4),
    "pool_type":      ["max", "avg"],
    "fc_hidden":      (0, 512),
    "lr":             (1e-4, 0.1),
    "wd":             (1e-6, 0.01),
    "optimizer":      ["sgd", "adam", "adamw"],
}

search = ArchSearch(
    train_fn=train_fn,
    search_space=search_space,
    backend="claude",  # uses Claude Haiku by default (~$0.01 per run)
)

search.run(max_evals=15)  # ~15 min on Colab GPU

print(f"\nBest config: {search.best_config}")
print(f"Best score:  {search.best_score:.4f}")
print(f"Best acc:    {search.best_accuracy:.4f}")

## Option 2: Just give it a model

Don't want to define a search space? Hand neuropt a PyTorch model — it introspects the architecture, discovers what's tunable (activations, dropout, batch norm, pooling), and builds the search space for you.

In [ ]:
model = torchvision.models.resnet18(weights=None, num_classes=10)
model.conv1 = nn.Conv2d(3, 64, 3, stride=1, padding=1, bias=False)  # adapt for 32x32
model.maxpool = nn.Identity()

def train_resnet(config):
    m = config["model"].to(DEVICE)
    opt_name = config.get("optimizer", "adamw")
    if opt_name == "sgd":
        opt = torch.optim.SGD(m.parameters(), lr=config["lr"], momentum=0.9, weight_decay=config["wd"])
    elif opt_name == "adam":
        opt = torch.optim.Adam(m.parameters(), lr=config["lr"], weight_decay=config["wd"])
    else:
        opt = torch.optim.AdamW(m.parameters(), lr=config["lr"], weight_decay=config["wd"])

    crit = nn.CrossEntropyLoss()
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=5)
    train_losses, val_losses, val_accuracies = [], [], []

    for _ in range(5):
        m.train()
        tl, tn = 0.0, 0
        for imgs, tgts in train_loader:
            imgs, tgts = imgs.to(DEVICE), tgts.to(DEVICE)
            opt.zero_grad()
            loss = crit(m(imgs), tgts)
            loss.backward(); opt.step()
            tl += loss.item() * imgs.size(0); tn += imgs.size(0)
        sched.step(); train_losses.append(tl / tn)

        m.eval()
        vl, vn, vc = 0.0, 0, 0
        with torch.no_grad():
            for imgs, tgts in val_loader:
                imgs, tgts = imgs.to(DEVICE), tgts.to(DEVICE)
                out = m(imgs)
                vl += crit(out, tgts).item() * imgs.size(0)
                vc += (out.argmax(1) == tgts).sum().item(); vn += imgs.size(0)
        val_losses.append(vl / vn); val_accuracies.append(vc / vn)

    return {"score": val_losses[-1], "accuracy": val_accuracies[-1],
            "train_losses": train_losses, "val_losses": val_losses, "val_accuracies": val_accuracies}

# neuropt figures out the search space automatically
search2 = ArchSearch.from_model(model, train_resnet, backend="claude")
search2.run(max_evals=10)

---

**What just happened?** neuropt used Claude Haiku (the cheapest Claude model) to read your training curves after each experiment and propose the next architecture to try. The summary above shows total API cost — typically a few cents for 15 evaluations.

**Next steps:**
- Try `max_evals=50` for better results
- Add `ml_context="..."` with domain knowledge about your task
- See the [docs](https://loevlie.github.io/neuropt/) for more options